## Bab 1: Tahap Persiapan & Instalasi Environment
Pada bab ini, kita akan menyiapkan seluruh library yang dibutuhkan untuk melakukan fine-tuning efisien menggunakan Unsloth pada GPU T4.

In [1]:
# Instalasi Unsloth versi cepat tanpa perlu build wheels lama
!pip install "unsloth[kaggle] @ git+https://github.com/unslothai/unsloth.git"
!pip install wandb
!pip install bitsandbytes --upgrade
print("✅ Semua library berhasil diinstal dengan cepat!")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-452jsc4k/unsloth_a84b9302599e46bdb3aeb9243cdf44aa
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-452jsc4k/unsloth_a84b9302599e46bdb3aeb9243cdf44aa
  Resolved https://github.com/unslothai/unsloth.git to commit f62c26e63d28c218c579c29a87e96d32ccde2eae
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 110.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 16.2 

## Bab 1.2: Otentikasi Akun Hugging Face & WandB
Silakan masukkan token Hugging Face Anda (Akses Write) dan API Key Weights & Biases Anda pada form di bawah ini agar proses upload model dan logging berjalan lancar.

In [2]:
import os
from huggingface_hub import login
import wandb
from kaggle_secrets import UserSecretsClient

# 1. Mengambil token secara aman dari Kaggle Secrets tanpa menampilkannya di teks kode
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("HF_TOKEN")
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

# Username Hugging Face tetap ditulis biasa tidak apa-apa karena bukan data rahasia
HF_USERNAME = "vikriahaikal" 

# 2. Eksekusi Login ke Hugging Face Hub
login(token=token)

# 3. Eksekusi Login ke Weights & Biases untuk logging training
wandb.login(key=wandb_key)

print("✅ Otentikasi Hugging Face dan WandB menggunakan Kaggle Secrets Berhasil!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vikrianalda (vikrianalda-universitas-serang-raya) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Otentikasi Hugging Face dan WandB menggunakan Kaggle Secrets Berhasil!


## Bab 2: Memuat & Menyiapkan Dataset Hukum
Kita memuat dataset dari Hugging Face Hub, menerapkan Chat Template Llama-3, dan menambahkan fungsi pembersihan (filter) token panjang maksimal 512 untuk mencegah kendala VRAM Out-of-Memory (OOM).

In [3]:
from datasets import load_dataset

# 1. Mengunduh dataset Alpaca GPT-4 Indonesia resmi
nama_repo_dataset = "Ichsan2895/alpaca-gpt4-indonesian" 

dataset = load_dataset(nama_repo_dataset)

print(f"✅ Berhasil memuat dataset dari Hugging Face!")
print(f"Jumlah data awal sebelum difilter: {len(dataset['train'])}")

README.md: 0.00B [00:00, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

✅ Berhasil memuat dataset dari Hugging Face!
Jumlah data awal sebelum difilter: 49969


## Bab 2.2: Terapkan Chat Template Llama-3 & Pembersihan Data (Kriteria Utama)
Menampilkan cetakan baris dataset terformat lengkap dengan token spesial sesuai kriteria kelulusan.

In [4]:
import os
import torch
import gc
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# 1. Load base model Meta-Llama-3-8B dengan 4-bit Quantization (Kriteria 3.1)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Meta-Llama-3-8B",
    max_seq_length = 512, 
    dtype = None,
    load_in_4bit = True,
    device_map = {"": 0}, 
)

# 2. Setup chat template Llama-3
tokenizer = get_chat_template(tokenizer, chat_template = "llama-3")

# 3. Fungsi mapping yang MENGUNGKAP MISTERI DATASET
def formatting_prompts_func(examples):
    # Di dataset Ichsan2895, HANYA ADA kolom 'input' dan 'output'!
    inputs  = examples["input"]
    outputs = examples["output"]
    texts = []
    
    for input_text, output_text in zip(inputs, outputs):
        text = tokenizer.apply_chat_template([
            {"role" : "user",   "content" : str(input_text)},
            {"role" : "assistant", "content" : str(output_text)},
        ], tokenize = False, add_generation_prompt = False)
        texts.append(text)
        
    return { "text" : texts }

# Mapping dataset
dataset['train'] = dataset['train'].map(formatting_prompts_func, batched = True)

# 4. FILTER DATASET (Membersihkan teks > 512 token agar kebal OOM)
dataset['train'] = dataset['train'].filter(lambda x: len(tokenizer(x['text'])['input_ids']) <= 512)

# BUKTI KRITERIA UTAMA: Cetak satu baris data terformat lengkap dengan token spesial
print("\n=== BUKTI DATASET TERFORMAT (KRITERIA DICODING) ===")
print(dataset['train'][0]['text'])
print("====================================================")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


Map:   0%|          | 0/49969 [00:00<?, ? examples/s]

Filter:   0%|          | 0/49969 [00:00<?, ? examples/s]


=== BUKTI DATASET TERFORMAT (KRITERIA DICODING) ===
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Saranlah slogan untuk kampanye daur ulang<|eot_id|><|start_header_id|>assistant<|end_header_id|>

1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."
2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."
3. "Ubah sampahmu menjadi harta karun - Daur ulang!"
4. "Daur ulang untuk siklus kehidupan."
5. "Simpan sumber daya, daur ulang lebih banyak."<|eot_id|>


## Bab 3: Parameter-Efficient Fine-Tuning (PEFT/LoRA)
Mengonfigurasi adapter parameter LoRA dengan efisiensi memori tingkat tinggi (Rank = 8) agar stabil berjalan pada GPU T4.

In [5]:
# Konfigurasi LoRA Adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, 
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("✅ Struktur LoRA Adapter berhasil disuntikkan ke Base Model!")

Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Struktur LoRA Adapter berhasil disuntikkan ke Base Model!


## Bab 4: Proses Pelatihan Model (Fine-Tuning)
Membagi dataset menjadi train/eval split dan mengeksekusi training dengan SFTConfig khusus untuk mencegah isu kegagalan pickling di library TRL.

In [6]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

# 1. Split Dataset (95% Train, 5% Test)
dataset_split = dataset['train'].train_test_split(test_size=0.05, seed=42)
train_data = dataset_split['train']
eval_data = dataset_split['test']

# 2. Setup Trainer Menggunakan SFTConfig Sesuai Best Practice Terbaru
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_data,
    eval_dataset = eval_data,
    dataset_text_field = "text",
    max_seq_length = 512,          
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(                    
        per_device_train_batch_size = 1, 
        gradient_accumulation_steps = 8, 
        per_device_eval_batch_size = 1,  
        warmup_steps = 5,
        max_steps = 800,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "paged_adamw_8bit", # Optimizer paged penyelamat memori
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb",
        run_name = "llama3-legal-bot-submission", 
        eval_strategy = "steps",         
        eval_steps = 200,
    ),
)

# 3. Eksekusi Pelatihan
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/41722 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2196 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 41,722 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 20,971,520 of 8,051,232,768 (0.26% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
200,1.339070,1.546652
400,1.368068,1.511401
600,1.294270,1.493311
800,1.351228,1.483628


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-800/tokenizer_config.json.


## Bab 5: Pengujian Model (Inference) & Ekspor ke Hugging Face Hub
Langkah akhir membuktikan performa model secara langsung dan menggabungkannya ke format 16-bit untuk di-upload langsung ke repositori publik Hugging Face.

In [7]:
# 1. Mengaktifkan Mode Cepat untuk Inference
FastLanguageModel.for_inference(model) 

# Pengujian dengan pertanyaan teks hukum
pertanyaan = "Jelaskan langkah-langkah yang harus dilakukan jika saya mengalami penipuan transaksi online!"

prompt = tokenizer.apply_chat_template([
    {"role": "user", "content": pertanyaan}
], tokenize = False, add_generation_prompt = True)

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

print(f"Pertanyaan Kelayakan: {pertanyaan}\n")
print("AI Sedang merumuskan jawaban...\n")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
jawaban = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]

print("=== OUTPUT JAWABAN MODEL ===")
print(jawaban.split("assistant\n")[-1].strip() if "assistant\n" in jawaban else jawaban)

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Pertanyaan Kelayakan: Jelaskan langkah-langkah yang harus dilakukan jika saya mengalami penipuan transaksi online!

AI Sedang merumuskan jawaban...



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


=== OUTPUT JAWABAN MODEL ===
1. Periksa rekening bank Anda: Pastikan bahwa Anda tidak melihat transaksi yang tidak diinginkan atau penurunan saldo yang tidak diinginkan.

2. Laporkan penipuan: Laporkan penipuan kepada bank Anda, situs web penjual, dan pihak keamanan online seperti FTC. Ini akan membantu menutup laporan penipuan dan mencegah orang lain dari jatuh ke dalam penipuan.

3. Batalkan transaksi: Jika transaksi telah selesai, Anda harus menghubungi bank Anda untuk membatalkan transaksi. Jika transaksi belum selesai, Anda harus menghubungi situs web penjual untuk membatalkan pesanan Anda.

4. Ubah kata sandi Anda: Ubah kata sandi Anda dan setel ulang kata sandi Anda di semua situs web yang Anda gunakan. Ini akan membantu memastikan bahwa penipu tidak dapat mengakses akun Anda lagi.

5. Gunakan teknologi keamanan: Pertimbangkan untuk menggunakan perangkat lunak keamanan atau layanan pelacakan untuk melacak aktivitas online Anda. Ini akan membantu


In [8]:
# 2. UPLOAD KE HUGGING FACE DENGAN METODE MERGED_16BIT (Kriteria Wajib Submission)
nama_repo = f"{HF_USERNAME}/llama3-legal-bot"

print(f"Sedang menggabungkan weight lora dan mengupload ke repositori: {nama_repo}...")

model.push_to_hub_merged(
    nama_repo, 
    tokenizer, 
    save_method = "merged_16bit", 
    token = token
)

print(f"\n🚀 SELESAI! Model Anda resmi mengudara secara permanen di:")
print(f"https://huggingface.co/{nama_repo}")

Sedang menggabungkan weight lora dan mengupload ke repositori: vikriahaikal/llama3-legal-bot...


config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in vikriahaikal/llama3-legal-bot/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...


Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [04:33<01:30, 90.70s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [04:53<00:00, 73.37s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/vikriahaikal/llama3-legal-bot`

🚀 SELESAI! Model Anda resmi mengudara secara permanen di:
https://huggingface.co/vikriahaikal/llama3-legal-bot
